<a id="top"></a>

# Demo: schemas as a teaching tool

```yaml
title: "Demo: schemas as a teaching tool"
keywords: parameter schema, loose vs tight schema, validation error, informative vs opaque, shape not truth, book_meeting, recoverable error, langchain, derived schema, teacher demo
estimated duration: 14
```

> **Lesson:** L08. Teacher demo — Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> **Model-agnostic by design.** Each schema *shape* is a **typed Python function**; `bind_tools` derives the JSON-Schema the model sees from the signature + docstring (we print it so you can read the contract). The model is a LangChain `ChatAnthropic`. The key loads through the config seam (`require_anthropic_key`); we never hard-code it.
>
> The point to land: **a tight schema converts ambiguity into validation errors the model can recover from; a loose schema pushes the ambiguity into a silent runtime guess.** And the error *shape* decides whether the model recovers — an error is a prompt for its next turn.

## Contents

- [1. Setup — loose vs tight book_meeting, and two error styles](#1-setup--loose-vs-tight-book_meeting-and-two-error-styles)
- [2. The loose schema — silent wrongness](#2-the-loose-schema--silent-wrongness)
- [3. Tight schema + opaque errors — a noisy, unhelpful loop](#3-tight-schema--opaque-errors--a-noisy-unhelpful-loop)
- [4. Tight schema + informative errors — failing productively](#4-tight-schema--informative-errors--failing-productively)
- [5. Schemas validate shape, not truth](#5-schemas-validate-shape-not-truth)
- [6. Takeaways](#6-takeaways)

## 1. Setup — loose vs tight book_meeting, and two error styles

One conceptual tool, `book_meeting`, in two schema shapes — each a **typed function**: **loose** (one free-form `details` string) and **tight** (typed, all-required fields with per-parameter descriptions and a `duration_minutes` bound). `bind_tools` derives the schema from each. The tight tool's validator comes in two styles — **informative** errors that name the field + constraint, and **opaque** errors that don't. One ambiguous prompt drives all three runs. Anchor model: **Claude Sonnet 4.6**. Live cells are gated on `LIVE`.

In [ ]:
from typing import Annotated, Any

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import Field

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"
LIVE = bool(get_settings().anthropic_api_key)


# Two schema SHAPES for the same conceptual tool, expressed as typed functions.
# bind_tools derives the JSON-Schema the model sees from each signature + docstring.
# The function bodies are stand-ins: the demo routes the actual call through the
# validators below, so it can show two ERROR styles for the same tight schema.
def book_meeting_loose(details: str) -> str:
    """Book a meeting."""
    return "ok"


def book_meeting_tight(
    attendee_email: str,
    start_iso: str,
    duration_minutes: Annotated[int, Field(ge=15, le=240)],
    title: str,
) -> str:
    """Book a calendar meeting. All fields are required and validated.

    Args:
        attendee_email: RFC 5322 email, e.g. 'priya@example.com'.
        start_iso: ISO 8601 start, e.g. '2024-11-05T14:00:00'.
        duration_minutes: integer between 15 and 240.
        title: short meeting title.
    """
    return "ok"


# Same NAME ('book_meeting'), two schemas. The tight tool keeps its per-field
# descriptions and the 15..240 bound, derived from the typed signature + docstring.
LOOSE_TOOL = StructuredTool.from_function(func=book_meeting_loose, name="book_meeting")
TIGHT_TOOL = StructuredTool.from_function(
    func=book_meeting_tight, name="book_meeting", parse_docstring=True
)


def validate_informative(args: dict[str, Any]) -> dict[str, Any]:
    """Validate tight-schema args; return a structured {error, field, message} or a booking."""
    email = str(args.get("attendee_email", ""))
    if "@" not in email:
        return {
            "error": "validation",
            "field": "attendee_email",
            "message": f"must be an email, got {email!r}",
        }
    minutes = args.get("duration_minutes")
    if not isinstance(minutes, int) or not (15 <= minutes <= 240):
        return {
            "error": "validation",
            "field": "duration_minutes",
            "message": f"must be an integer 15..240, got {minutes!r}",
        }
    return {"booked": True, "with": email, "minutes": minutes}


def validate_opaque(args: dict[str, Any]) -> dict[str, Any]:
    """Same checks, but every failure collapses to a useless generic message."""
    res = validate_informative(args)
    if "error" in res:
        return {"error": "bad input"}  # no field, no constraint
    return res


def show_turn(reply: AIMessage) -> None:
    """Print a reply's text and any tool calls (name + args)."""
    if reply.text:
        print(f"  text      {reply.text!r}")
    for call in reply.tool_calls:
        print(f"  tool call  {call['name']}  args={call['args']}")


# Ambiguous on purpose: no email, vague "afternoon", relative date.
PROMPT = "Book a 90-minute design review with Priya next Tuesday afternoon."

if LIVE:
    model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)

print("setup ready (LIVE =", LIVE, ")")

**The tight contract, derived from the typed function.** You wrote a function; the model receives this JSON-Schema. Every field typed, all required, `duration_minutes` bounded, a description on each — the whole "tight schema" idea, expressed as a signature + docstring.

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

# The contract the model receives for the TIGHT tool — derived from the typed
# function + docstring. Note the enum-free but bounded duration (minimum/maximum),
# the required list, and a description on every field. No key needed to see it.
print(json.dumps(convert_to_openai_tool(TIGHT_TOOL), indent=2))

[↑ Back to top](#top)

## 2. The loose schema — silent wrongness

Run the ambiguous prompt against the **loose** schema. The model packs everything into the `details` string and the tool 'succeeds'. Whether the meeting was booked *correctly* — with whose email, at what exact time — is **impossible to tell** from the conversation. This is the worst outcome: it looks like it worked.

In [ ]:
if LIVE:
    reply = model.bind_tools([LOOSE_TOOL]).invoke([HumanMessage(PROMPT)])
    show_turn(reply)
    if reply.tool_calls:
        print("\nthe tool got ONE blob:", reply.tool_calls[0]["args"])
        print("booked correctly? impossible to tell from here.")
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 3. Tight schema + opaque errors — a noisy, unhelpful loop

Run the same prompt against the **tight** schema with **opaque** errors. The model guesses each field and submits; the validator returns `{'error': 'bad input'}`. We hand that back and let the model try again — it typically retries with *another* guess, still opaque, still wrong. We run two rounds to show the loop.

In [ ]:
if LIVE:
    messages = [HumanMessage(PROMPT)]
    bound = model.bind_tools([TIGHT_TOOL])
    for round_num in range(2):
        reply = bound.invoke(messages)
        print(f"--- round {round_num + 1} ---")
        show_turn(reply)
        if not reply.tool_calls:
            break
        call = reply.tool_calls[0]
        result = validate_opaque(call["args"])
        print("  tool returned:", result)
        messages.append(reply)
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 4. Tight schema + informative errors — failing productively

Same prompt, same tight schema, but now **informative** errors. The model submits, gets a structured error naming the offending field, and on the next turn either **asks the user** for the missing detail (Priya's email) or **fixes the field** and retries. The error taught the model what to do.

In [ ]:
if LIVE:
    messages = [HumanMessage(PROMPT)]
    bound = model.bind_tools([TIGHT_TOOL])
    for round_num in range(3):
        reply = bound.invoke(messages)
        print(f"--- round {round_num + 1} ---")
        show_turn(reply)
        if not reply.tool_calls:
            print("  (model stopped calling — asked the user or finished)")
            break
        call = reply.tool_calls[0]
        result = validate_informative(call["args"])
        print("  tool returned:", result)
        messages.append(reply)
        messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        if "booked" in result:
            break
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 5. Schemas validate shape, not truth

A subtle trap to name out loud: the model may guess `priya@example.com` so confidently that it **passes** email-format validation — even though no such user may exist. The tight schema catches *malformed* arguments; it does **not** catch *false* ones. That gap is exactly why **runtime errors** ([L0809](L0809_lecture.ipynb)) are the second line of defense.

In [ ]:
fake_but_well_formed = {
    "attendee_email": "priya@example.com",  # valid shape, possibly nonexistent person
    "start_iso": "2024-11-05T14:00:00",
    "duration_minutes": 90,
    "title": "Design review",
}
print("validation says:", validate_informative(fake_but_well_formed))
print("note: the shape is valid even if the address is fake — shape != truth.")

[↑ Back to top](#top)

## 6. Takeaways

- The **loose** schema *appears* to work and is the worst outcome (silent wrongness). The tight schema with **opaque** errors fails noisily but unhelpfully. The tight schema with **informative** errors fails *productively* — the model learns from each turn.
- A tight schema — required fields, narrow types, per-field constraints — turns the model's fuzziness ([L02](../L02/L0201_intro.md)) into **recoverable validation errors**.
- An **error message is a prompt** for the model's next turn — write it like a system message, not a Python traceback. The [L0808 lab](L0808_lab_empty.ipynb) drills exactly this rewrite, offline.
- Schemas validate **shape, not truth** — the bridge to [L0809](L0809_lecture.ipynb)'s runtime errors and side effects.

[↑ Back to top](#top)